In [1]:
import json
from src.db import DbConnection
from src.server.models.dms_constants import DmsConstants
from src.server.models.dms_model import DmsCache
from src.runtime_constants import rr


In [2]:
survey_control_guid = "a236b79f-6d2e-4f23-ad92-d018cb518346"
dcm_df_guid = "be8b281a-5fa9-4308-8b23-0f1ff92f1070"

In [20]:
conn.close()

In [3]:
#conn =  DbConnection(rr.ENV_PROD, True)
conn =  DbConnection()

In [4]:
# load sheets

In [5]:
def load_data(control_guid: str):


    row_config = conn.select_one("""
    SELECT cfg.config
      FROM survey_v2.survey_control sc,
           survey_v2.survey_config cfg
     WHERE sc.guid = %s
       AND sc.survey_config = cfg.guid
    """, [control_guid])

    rows = conn.select_all("""
    SELECT si.guid, sw.state, si.client_data, si.server_data, si.ext_data
      FROM survey_v2.survey_instance si,
           survey_v2.survey_whitelist sw,
           survey_v2.survey_control sc
     WHERE sc.guid = %s
       AND sw.survey_control = sc.id
       AND si.guid = sw.gid
       AND sw.state = 4
    """, [control_guid])

    sheet_config = DmsCache.get_from_cache(conn=conn, guid=DmsConstants.survey_v2_config)["data"]
    sheet_texts = DmsCache.get_from_cache(conn=conn, guid=DmsConstants.survey_v2_texts)["data"]

    def_roles = [
        {"value": 3, "id": "role__hospitality"},
        {"value": 4, "id": "role__automotive"},
        {"value": 7, "id": "role__banking"},
        {"value": 10, "id": "role__other"},
    ]

    return {
        "config": json.loads(row_config["config"]),
        "surveys": rows,
        "sheet_config": sheet_config,
        "sheet_texts": sheet_texts,
        "definitions": {
            "role": def_roles
        }
    }

In [6]:
loaded_data = load_data(survey_control_guid)
sheet_config = loaded_data["sheet_config"]
sheet_texts = loaded_data["sheet_texts"]
surveys = loaded_data["surveys"]

In [7]:
def get_text(code: str):
    for row in sheet_texts["texts"]:
        if row["id"] == code:
            return row["EN"]
    return code

def get_config_answers(page: str) -> []:
    res = []
    for row in sheet_config[page]:
        if row["param"] == "answer":
            res.append(row)
    return res

def get_config_items(page: str) -> []:
    res = []
    for row in sheet_config[page]:
        if row["param"] == "item":
            res.append(row)
    return res

def get_answer_text(page: str, answer_id: str) -> str:
    answers = get_config_answers(page)
    for it in answers:
        if it["value"] == answer_id:
            return get_text(it["text"])
    return ""

def get_competitors_titles() -> []:
    res = []
    for row in sheet_config["apps"]:
        if row["title"] not in res:
            res.append(row["title"])
    return res

def get_competitor_title(app_id: str) -> str:
    for row in sheet_config["apps"]:
        if row["id"] == app_id:
            return row["title"]
    return None

def get_gender_tag(gender: str):
    return f"gender__{gender}"

def get_age_tag(age_from: int, age_to: int):
    return f"age__{age_from}_{age_to}"

def get_role_tag(role: int):
    for it in loaded_data["definitions"]["role"]:
        if it["value"] == role:
            return f"role__{it['id']}"
    raise Exception(f"Unknow role {role}")

In [8]:

import statistics

data = []
dcm_nodes = []

survey_info_rows = []
survey_tags_rows = []
survey_flags_rows = []
survey_competitors_rows = []
survey_genres_dcm_rows = []
survey_topics_dcm_rows = []

for row in surveys:
    try:
        guid = row["guid"]
        survey_genres_dcm_stats = {}
        survey_topics_dcm_stats = {}

        survey_info_row = {
            "survey_instance": guid,
            "survey_genres_dcm_stats": survey_genres_dcm_stats,
            "survey_topics_dcm_stats": survey_topics_dcm_stats
        }
        survey_tags_row = []
        survey_flags_row = []
        survey_competitors_row = []
        survey_genres_dcm_row = []
        survey_topics_dcm_row = []

        data_row = {}
        genres_dcm_rows = []

        client_data = json.loads(row["client_data"])
        server_data = json.loads(row["server_data"])
        ext_data = json.loads(row["ext_data"])
        pages = client_data["pages"]

        data_row["guid"] = guid

        # time
        data_row["time"] = 0
        genres_time = -1
        topics_time = -1
        if "server_time" in ext_data:
            min = None
            max = None
            for it in ext_data["server_time"]:
                if it["id"] == "intro":
                    min = it["export"]["f"]
                if it["id"] == "end":
                    max = it["export"]["l"]
                if it["id"] == "genres" or it["id"] == "topics":
                    min_g = None
                    max_g = None
                    if "export" in it and "f" in it["export"]:
                        min_g = it["export"]["f"]
                    if "submit" in it and "l" in it["submit"]:
                        max_g = it["submit"]["l"]
                    if min_g is not None and max_g is not None:
                        if it["id"] == "genres":
                            genres_time = max_g - min_g
                        elif it["id"] == "topics":
                            topics_time = max_g - min_g
            if min is not None and max is not None:
                data_row["time"] = max - min

        # gender
        page = "gender"
        data_row[page] = None
        data_row[page] = pages[page][page]
        survey_info_row[page] = pages[page][page]

        # age
        page = "age"
        data_row[page] = None
        age: str = pages[page][page]
        age_arr = age.split("__")
        age_val = None
        if age_arr[0] == "-1":
            age_val = 18
        elif age_arr[1] == "-1":
            age_val = 65
        else:
            age_val = (int(age_arr[1]) + int(age_arr[0])) // 2
        data_row[page] = int(age_val)
        survey_info_row[page] = int(age_val)

        # role
        page = "role"
        data_row[page] = None
        text = get_answer_text(page, pages[page][page])
        data_row[page] = text
        survey_info_row[page] = int(pages[page][page])

        # favorite_game
        page = "favorite_game"
        data_row[page] = pages[page][page]
        survey_info_row[page] = pages[page][page]

        # spending
        page = "spending"
        for it in get_config_answers(page):
            spending_col = f"{page}__{it['value']}__{it['value_2']}"
            data_row[spending_col] = 0
        spending_col = f"{page}__{pages[page][page]}"
        spending_arr = pages[page][page].split("__")
        if spending_arr[0] == "-1":
            spending_arr[0] = "0"
        elif spending_arr[1] == "-1":
            spending_arr[1] = "1000"
        spending_val = (int(spending_arr[0]) + int(spending_arr[1])) // 2
        data_row[spending_col] = 1
        survey_info_row[page] = spending_val

        # playing time
        page = "playing_time"
        for it in get_config_answers(page):
            playing_time_col = f"{page}__{it['value']}__{it['value_2']}"
            data_row[playing_time_col] = 0
        playing_time_col = f"{page}__{pages[page][page]}"
        data_row[playing_time_col] = 1
        playing_arr = pages[page][page].split("__")
        if playing_arr[0] == "-1" and playing_arr[1] == "-1":
            playing_arr[0] = "0"
            playing_arr[1] = "0"
        elif playing_arr[0] == "-1":
            playing_arr[0] = "0"
        elif playing_arr[1] == "-1":
            playing_arr[1] = "15"
        playing_val = (int(playing_arr[0]) + int(playing_arr[1])) // 2
        survey_info_row[page] = playing_val

        # loyalty
        page = "loyalty"
        for it in get_config_answers(page):
            loyalty_col = f"{page}__{it['value']}__{it['value_2']}"
            data_row[loyalty_col] = 0
        loyalty_col = f"{page}__{pages[page][page]}"
        data_row[loyalty_col] = 1
        arr = pages[page][page].split("__")
        if arr[0] == "-1" and arr[1] == "-1":
            pass
        elif arr[0] == "-1":
            arr[0] = "0"
        elif arr[1] == "-1":
            arr[1] = "5"
        val = (int(arr[0]) + int(arr[1])) // 2
        survey_info_row[page] = val

        # devices
        for it in get_config_answers("devices"):
            data_row[it["value"]] = 0
        for it in pages["devices"]["devices"]:
            data_row[it] = 1
            survey_flags_row.append({
                "survey_instance": guid,
                "tag_id": it
            })

        # genres
        data_row["genres_time"] = genres_time
        for it in get_config_items("genres"):
            data_row[it["value"]] = -1
        for genre in pages["genres"]:
            data_row[genre] = int(pages["genres"][genre])
            survey_tags_row.append({
                "survey_instance": guid,
                "tag_id": genre,
                "tag_value": int((int(pages["genres"][genre]) - 1) / 3 * 100)
            })

        # topics
        data_row["topics_time"] = topics_time
        for it in get_config_items("topics"):
            data_row[it["value"]] = -1
        for topic in pages["topics"]:
            data_row[topic] = int(pages["topics"][topic])
            survey_tags_row.append({
                "survey_instance": guid,
                "tag_id": topic,
                "tag_value": int((int(pages["topics"][topic]) - 1) / 3 * 100)
            })

        # needs
        for it in get_config_items("needs"):
            data_row[it["value"]] = -1
        for need in pages["needs"]:
            data_row[need] = int(pages["needs"][need])
            survey_tags_row.append({
                "survey_instance": guid,
                "tag_id": need,
                "tag_value": int((int(pages["needs"][need]) - 1) / 2 * 100)
            })

        # behaviors
        def migr_behaviors(old: str) -> str:
            if old == "behaviors__experiment":
                return "behaviors__explore"
            if old == "behaviors__discover":
                return "behaviors__explore"
            if old == "behaviors__experience":
                return "behaviors__immerse"
            if old == "behaviors__combat":
                return "behaviors__destroy"
            if old == "behaviors__complete":
                return "behaviors__utilize"
            if old == "behaviors__collect":
                return "behaviors__utilize"
            if old == "behaviors__empower":
                return "behaviors__cultivate"
            if old == "behaviors__solve":
                return "behaviors__improvise"
            if old == "behaviors__cooperate":
                return "behaviors__bond"
            if old == "behaviors__compete":
                return "behaviors__challenge"
            return old

        behaviors_values = {}
        for it in get_config_items("behaviors"):
            data_row[migr_behaviors(it["value"])] = -1
        for behavior in pages["behaviors"]:
            behavior_new = migr_behaviors(behavior)
            if behavior_new not in behaviors_values:
                behaviors_values[behavior_new] = []
            behaviors_values[behavior_new].append(int(pages["behaviors"][behavior]))

        for tag_id in behaviors_values:
            avg = statistics.mean(behaviors_values[tag_id])
            data_row[tag_id] = int(avg)
            survey_tags_row.append({
                "survey_instance": guid,
                "tag_id": tag_id,
                "tag_value": int((avg - 1) / 2 * 100)
            })

        # multiplayer
        for it in get_config_items("multiplayer"):
            data_row[it["value"]] = -1
        if "multiplayer" in pages:
            for multiplayer in pages["multiplayer"]:
                data_row[multiplayer] = int(pages["multiplayer"][multiplayer])
                survey_tags_row.append({
                    "survey_instance": guid,
                    "tag_id": multiplayer,
                    "tag_value": int(pages["multiplayer"][multiplayer])
                })

        # routine
        for it in get_config_answers("routine"):
            data_row[it["value"]] = 0
        arr = pages["routine"]["routine"]
        for it in arr:
            data_row[it] = 1
            survey_flags_row.append({
                "survey_instance": guid,
                "tag_id": it,
            })

        # hobbies
        for it in get_config_answers("hobbies"):
            data_row[it["value"]] = -1
        if "hobbies" in pages:
            for it in get_config_answers("hobbies"):
                data_row[it["value"]] = 0
            arr = pages["hobbies"]["hobbies"]
            for it in arr:
                data_row[it] = 1
                survey_flags_row.append({
                    "survey_instance": guid,
                    "tag_id": it
                })

        # movies
        for it in get_config_answers("movies"):
            data_row[it["value"]] = -1
        if "movies" in pages:
            for it in get_config_answers("movies"):
                data_row[it["value"]] = 0
            arr = pages["movies"]["movies"]
            for it in arr:
                data_row[it] = 1
                survey_flags_row.append({
                    "survey_instance": guid,
                    "tag_id": it
                })

        # socials
        for it in get_config_answers("socials"):
            data_row[it["value"]] = -1
        if "socials" in pages:
            for it in get_config_answers("socials"):
                data_row[it["value"]] = 0
            arr = pages["socials"]["socials"]
            for it in arr:
                data_row[it] = 1
                survey_flags_row.append({
                    "survey_instance": guid,
                    "tag_id": it
                })

        # competitors
        for it in get_competitors_titles():
            data_row[f"C: {it}"] = -1
        if "competitors" in pages:
            for competitor in pages["competitors"]:
                title = get_competitor_title(competitor)
                data_row[f"C: {title}"] = int(pages["competitors"][competitor])
                survey_competitors_row.append({
                    "survey_instance": guid,
                    "app_id": competitor,
                    "value": int(pages["competitors"][competitor])
                })

        # genres_dcm
        for i in range(5):
            col_it = f"Genre DCM: it {i + 1}"
            data_row[f"{col_it} - genre 1"] = None
            data_row[f"{col_it} - genre 2"] = None
            data_row[f"{col_it} - core 1"] = None
            data_row[f"{col_it} - core 2"] = None
            data_row[f"{col_it} - chosen"] = None
        if "genres_dcm" in pages:
            server_dcm = server_data["pages"]["genres_dcm"]
            ix = 0
            for it in server_dcm["dcm"]:
                ix += 1
                genre1 = it["genres"][0]
                dcm_nodes.append(genre1)
                genre2 = None
                if len(it["genres"]) == 2:
                    genre2 = it["genres"][1]
                    dcm_nodes.append(genre2)
                core1 = it["choices"][0]
                core2 = it["choices"][1]
                chosen = pages["genres_dcm"][str(it["id"])]
                col_it = f"Genre DCM: it {it['id']}"
                data_row[f"{col_it} - genre 1"] = genre1
                data_row[f"{col_it} - genre 2"] = genre2
                data_row[f"{col_it} - core 1"] = core1
                data_row[f"{col_it} - core 2"] = core2
                data_row[f"{col_it} - chosen"] = chosen
                dcm_nodes.append(core1)
                dcm_nodes.append(core2)
                if chosen == "none":
                    chosen = None

                if core1 not in survey_genres_dcm_stats:
                    survey_genres_dcm_stats[core1] = {
                        "count": 0,
                        "chosen": 0
                    }
                if core2 not in survey_genres_dcm_stats:
                    survey_genres_dcm_stats[core2] = {
                        "count": 0,
                        "chosen": 0
                    }
                survey_genres_dcm_stats[core1]["count"] += 1
                survey_genres_dcm_stats[core2]["count"] += 1
                if chosen == core1:
                    survey_genres_dcm_stats[core1]["chosen"] += 1
                if chosen == core2:
                    survey_genres_dcm_stats[core2]["chosen"] += 1

                survey_genres_dcm_row.append({
                    "survey_instance": guid,
                    "dcm_order": ix,
                    "genre_1": genre1,
                    "genre_2": genre2,
                    "core_1": core1,
                    "core_2": core2,
                    "chosen": chosen
                })

        # topics_dcm
        for i in range(5):
            col_it = f"Topics DCM: it {i + 1}"
            data_row[f"{col_it} - topic"] = None
            data_row[f"{col_it} - f1 a"] = None
            data_row[f"{col_it} - f1 b"] = None
            data_row[f"{col_it} - f2 a"] = None
            data_row[f"{col_it} - f2 b"] = None
            data_row[f"{col_it} - chosen"] = None
        if "topics_dcm" in pages:
            server_dcm = server_data["pages"]["topics_dcm"]
            ix = 0
            for it in server_dcm["dcm"]:
                ix += 1
                topic = it["topic"]
                dcm_nodes.append(topic)
                f1a = it["choices"][0]["items"][0]
                dcm_nodes.append(f1a)
                f1b = it["choices"][0]["items"][1]
                dcm_nodes.append(f1b)
                f2a = it["choices"][1]["items"][0]
                dcm_nodes.append(f2a)
                f2b = it["choices"][1]["items"][1]
                dcm_nodes.append(f2b)
                chosen = pages["topics_dcm"][str(it["id"])]
                chosen_arr = chosen.split("__")
                chosen_a = None
                chosen_b = None
                if chosen_arr[0] == "none":
                    chosen_final = "none"
                else:
                    chosen_final = f"f{int(chosen_arr[1]) + 1}"
                    if int(chosen_arr[1]) == 0:
                        chosen_a = f1a
                        chosen_b = f1b
                    else:
                        chosen_a = f2a
                        chosen_b = f2b
                col_it = f"Topics DCM: it {it['id']}"
                data_row[f"{col_it} - topic"] = topic
                data_row[f"{col_it} - f1 a"] = f1a
                data_row[f"{col_it} - f1 b"] = f1b
                data_row[f"{col_it} - f2 a"] = f2a
                data_row[f"{col_it} - f2 b"] = f2b
                data_row[f"{col_it} - chosen"] = chosen_final

                for f in [f1a, f1b, f2a, f2b]:
                    if f not in survey_topics_dcm_stats:
                        survey_topics_dcm_stats[f] = {
                            "count": 0,
                            "chosen": 0
                        }
                    survey_topics_dcm_stats[f]["count"] += 1
                    survey_topics_dcm_stats[f]["count"] += 1

                for c in [chosen_a, chosen_b]:
                    if c is not None:
                        survey_topics_dcm_stats[c]["chosen"] += 1

                survey_topics_dcm_row.append({
                    "survey_instance": guid,
                    "dcm_order": ix,
                    "topic": topic,
                    "feature_1a": f1a,
                    "feature_1b": f1b,
                    "feature_2a": f2a,
                    "feature_2b": f2b,
                    "chosen_a": chosen_a,
                    "chosen_b": chosen_b,
                })

        survey_info_rows.append(survey_info_row)
        for it in survey_tags_row:
            survey_tags_rows.append(it)
        for it in survey_flags_row:
            survey_flags_rows.append(it)
        for it in survey_genres_dcm_row:
            survey_genres_dcm_rows.append(it)
        for it in survey_topics_dcm_row:
            survey_topics_dcm_rows.append(it)
        for it in survey_competitors_row:
            survey_competitors_rows.append(it)

        data.append(data_row)
    except Exception as e:
        print(f"Error: {guid}: {str(e)}")
        continue

#data
print("DONE")


Error: ecd87717-63a5-44b7-8ccc-19eb5003238b: 'gender'
Error: f6bdbec7-d844-4ffa-b958-6659b9279d82: 'routine'
Error: 319c7844-4c77-40be-8ccf-a26a2a49c203: 'routine'
Error: 48c86e59-5182-4bec-ae40-4f27ed58b760: 'routine'
DONE


In [9]:
from src.server.models.map_tag_id import TagsMapper

row = conn.select_one_or_none("""
SELECT m.id
  FROM survey_data.survey_meta m
 WHERE m.survey_control_guid = %s
""", [survey_control_guid])

if row is not None:
    meta_id = row["id"]

    conn.query("""
    DELETE FROM survey_data.survey_info WHERE survey_meta_id = %s
    """, [meta_id])

    conn.query("""
    DELETE FROM survey_data.survey_tags WHERE survey_meta_id = %s
    """, [meta_id])

    conn.query("""
    DELETE FROM survey_data.survey_flags WHERE survey_id = %s
    """, [meta_id])

    conn.query("""
    DELETE FROM survey_data.survey_competitors WHERE survey_meta_id = %s
    """, [meta_id])

    conn.query("""
    DELETE FROM survey_data.survey_genres_dcm WHERE survey_meta_id = %s
    """, [meta_id])

    conn.query("""
    DELETE FROM survey_data.survey_topics_dcm WHERE survey_meta_id = %s
    """, [meta_id])

    conn.query("""
    UPDATE survey_data.survey_meta SET version = 2
    WHERE id = %s
    """, [meta_id])
else:
    meta_id = conn.insert("""
    INSERT INTO survey_data.survey_meta (survey_control_guid, version) VALUES (%s, 2)
    """, [survey_control_guid])

conn.commit()

q_data = []
for row in survey_info_rows:
    q_data.append((meta_id, row["survey_instance"], row["age"], row["gender"], row["role"], row["favorite_game"], row["spending"], row["playing_time"], row["loyalty"]))
conn.bulk("""
INSERT INTO survey_data.survey_info (survey_meta_id, survey_instance, age, gender, role, favorite_game, spending, playing_time, loyalty) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
""", q_data)

conn.commit()

q_data = []
for row in survey_competitors_rows:
    q_data.append((meta_id, row["survey_instance"], row["app_id"], row["value"]))
conn.bulk("""
INSERT INTO survey_data.survey_competitors (survey_meta_id, survey_instance, app_id, value) VALUES (%s, %s, %s, %s)
""", q_data)

conn.commit()

q_data = []
for row in survey_genres_dcm_rows:
    q_data.append((meta_id, row["survey_instance"], row["dcm_order"], row["genre_1"], row["genre_2"], row["core_1"], row["core_2"], row["chosen"]))
conn.bulk("""
INSERT INTO survey_data.survey_genres_dcm (survey_meta_id, survey_instance, dcm_order, genre_1, genre_2, core_1, core_2, chosen) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
""", q_data)

conn.commit()

q_data = []
for row in survey_topics_dcm_rows:
    q_data.append((meta_id, row["survey_instance"], row["dcm_order"], row["topic"], row["feature_1a"], row["feature_1b"], row["feature_2a"], row["feature_2b"], row["chosen_a"], row["chosen_b"]))
conn.bulk("""
INSERT INTO survey_data.survey_topics_dcm (survey_meta_id, survey_instance, dcm_order, topic, feature_1a, feature_1b, feature_2a, feature_2b, chosen_a, chosen_b) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", q_data)

conn.commit()

q_data = []
map_tags = TagsMapper.get_map_tag_id(conn)
for row in survey_tags_rows:
    q_data.append((meta_id, row["survey_instance"], row["tag_id"], row["tag_value"]))
conn.bulk("""
INSERT INTO survey_data.survey_tags (survey_meta_id, survey_instance, tag_id, tag_value)
VALUES (%s, %s, %s, %s)
""", q_data)

conn.commit()

q_data = []
map_tags = TagsMapper.get_map_tag_id(conn)
for row in survey_flags_rows:
    q_data.append((meta_id, row["survey_instance"], row["tag_id"]))
conn.bulk("""
INSERT INTO survey_data.survey_flags (survey_id, survey_instance, tag_id)
VALUES (%s, %s, %s)
""", q_data)

conn.commit()

In [14]:
from src.server.models.dms_model import DmsModel
from scipy import stats
import numpy as np

survey_meta_id = conn.select_one("""
SELECT m.id
  FROM survey_data.survey_meta m
 WHERE m.survey_control_guid = %s
""", [survey_control_guid])["id"]
df = DmsModel.read_df_from_external_data(conn, dcm_df_guid)

for col in df.columns:
    if col == "survey_instance" or col == "index" or col == "base":
        continue
    df_tmp = df[col][np.abs(stats.zscore(df[col])) < 3]
    min_val = np.min(df_tmp.to_numpy())
    max_val = np.max(df_tmp.to_numpy())
    normalized_col = "n___" + col
    df[normalized_col] = df.apply(lambda r: (r[col] - min_val) / (max_val - min_val), axis=1)
    df[normalized_col].clip(lower=0, upper=1, inplace=True)

q_data = []
for index, row in df.iterrows():
    for col in df.columns:
        if "n__" in col:
            val = row[col]
            val = int(val * 100)
            if val >= 80:
                val = 100
            elif val <= 20:
                val = 0
            else:
                val = 50
            q_data.append((survey_meta_id, row["survey_instance"], col.replace("n___", ""), val))

conn.bulk("""
INSERT INTO survey_data.survey_tags (survey_meta_id, survey_instance, tag_id, tag_value) VALUES (%s, %s, %s, %s)
""", q_data)
conn.commit()


KeyboardInterrupt: 

SURVEY POSTPROCESS

In [2]:
from src.server.models.platform_values.platform_values_pre_calc import PlatformValuesPreCalc
from src.server.models.platform_values.platform_values_utils import PlatformValuesUtils
from src.server.models.survey_data_model import SurveyDataModel
from src.db.mysql import DbConnection
from src.runtime_constants import rr

#rr.set_env(rr.ENV_DEV)
rr.set_env(rr.ENV_PROD, True)
conn = DbConnection()

survey_control_guid = "a236b79f-6d2e-4f23-ad92-d018cb518346"
SurveyDataModel.postprocess(conn=conn, survey_control_guid=survey_control_guid)
PlatformValuesUtils.recreate_platform_values_tables(conn)
PlatformValuesPreCalc(conn=conn).run_cod(run_tam=False)

conn.commit()
conn.close()

create_survey_tags_w
Page: 0
Page: 1
Page: 2
Page: 3
Page: 4
Page: 5
Page: 6
Page: 7
Page: 8
Page: 9
Page: 10
Page: 11
Page: 12
Page: 13
Page: 14
Page: 15
Page: 16
Page: 17
Page: 18
Page: 19
Page: 20
Page: 21
Page: 22
Page: 23
Page: 24
Page: 25
Page: 26
Page: 27
create_fake_concepts_tam


In [12]:
conn.close()

In [37]:
values = []

cols = []
for k in data[0]:
    cols.append(k)

values.append(cols)

for it in data:
    row = []
    for c in cols:
        row.append(it[c])
    values.append(row)

for it in values:
    print(len(it))

313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313
313


In [38]:
from src import external_api

external_api.write_sheet("1Kl3VFIHkWDQNW5hCvuXlXA7AOc3Hh-jRlQ5kVS8nazo", "data", values)

{'spreadsheetId': '1Kl3VFIHkWDQNW5hCvuXlXA7AOc3Hh-jRlQ5kVS8nazo',
 'updatedRange': 'data!A1:LA1971',
 'updatedRows': 1971,
 'updatedColumns': 313,
 'updatedCells': 591800}

In [39]:
conn = DbConnection()
product_sheet = DmsCache.get_json(conn, guid=DmsConstants.platform_guid)["Product"]
conn.close()

nodes_in_dcm = {}
for row in product_sheet:
    if row["ID"] is None or row["ID"] == "":
        continue
    sub = row["Subcategory"]
    if sub == "Genre" or sub == "Core" or sub == "Entities" or sub == "Eras" or sub == "Environment" or sub == "Roles" or sub == "Domains" or sub == "Focus" or sub == "Topics":
        nodes_in_dcm[row["ID"]] = 0

for it in dcm_nodes:
    nodes_in_dcm[it] += 1

nodes_in_dcm

values = [["node", "count"]]
for k in nodes_in_dcm:
    values.append([k, nodes_in_dcm[k]])

values

external_api.write_sheet("1Kl3VFIHkWDQNW5hCvuXlXA7AOc3Hh-jRlQ5kVS8nazo", "dcm_nodes", values)

{'spreadsheetId': '1Kl3VFIHkWDQNW5hCvuXlXA7AOc3Hh-jRlQ5kVS8nazo',
 'updatedRange': 'dcm_nodes!A1:B218',
 'updatedRows': 218,
 'updatedColumns': 2,
 'updatedCells': 436}

COMPLEXITY AND MULTIPLAYER TAGS

In [1]:
import json
from src.db.mysql import DbConnection
from src.runtime_constants import rr
#rr.set_env(rr.ENV_DEV)
rr.set_env(rr.ENV_PROD, True)
conn = DbConnection()
survey_id = 3

for tag_id_int in [11, 12, 13, 14]:

    query = f"""
    DELETE FROM survey_data.survey_tags st
    WHERE st.tag_id = (
    SELECT mt.tag_id FROM app.map_tag_id mt WHERE mt.id = {tag_id_int})
    AND st.survey_meta_id = {survey_id}
    """
    conn.query(query)
    conn.commit()

    query = f"""
    INSERT INTO survey_data.survey_tags (survey_meta_id, tag_id, survey_instance, tag_value)
    SELECT z2.survey_meta_id, z2.tag_id, z2.survey_instance,
    round(avg(z2.tag_value)) as tag_value
      FROM (
    SELECT z1.survey_meta_id,
    z1.tag_id, z1.tag_value, z1.survey_instance,
    COUNT(1) OVER (PARTITION BY z1.survey_instance) AS total_num,
    ROW_NUMBER() OVER (PARTITION BY z1.survey_instance ORDER BY tag_value) AS row_num
    FROM (
    SELECT sc.survey_meta_id, mt.tag_id,
    CASE WHEN sc.value = 5 THEN 100
         WHEN sc.value = 4 THEN 100
         WHEN sc.value = 1 THEN 0
         ELSE 50
         END AS tag_value,
     sc.survey_instance
      FROM survey_data.survey_competitors sc,
           app.map_app_id m,
           tagged_data.tags t,
           app.map_tag_id mt
     WHERE m.app_id = sc.app_id
       AND t.app_id_int = m.id
       AND t.tag_id_int = {tag_id_int}
       AND mt.id = t.tag_id_int
       AND sc.survey_meta_id = {survey_id}
    ) z1
    ) z2
    WHERE z2.row_num IN (FLOOR((z2.total_num + 1) / 2), FLOOR((z2.total_num + 2) / 2) )
    GROUP BY z2.survey_meta_id, z2.tag_id, z2.survey_instance;
    """
    conn.query(query)
    conn.commit()

query = """
DELETE FROM survey_data.survey_tags st
WHERE st.tag_id = (
SELECT mt.tag_id FROM app.map_tag_id mt WHERE mt.id = 15);
"""
conn.query(query)
conn.commit()
query = """
INSERT INTO survey_data.survey_tags (survey_meta_id, tag_id, tag_value, survey_instance)
SELECT st.survey_meta_id, mt.tag_id,
FLOOR((st.tag_value - 1) / 3 * 100) AS tag_value,
 st.survey_instance
  FROM survey_data.survey_tags st,
       app.map_tag_id mt
 WHERE st.tag_id = 'multiplayer__single'
   AND mt.id = 15;
"""
conn.query(query)
conn.commit()

for tag_id_int in [16, 17, 18, 19]:
    query = f"""
    DELETE FROM survey_data.survey_tags st
    WHERE st.tag_id = (
    SELECT mt.tag_id FROM app.map_tag_id mt WHERE mt.id = {tag_id_int})
    AND survey_meta_id = {survey_id}
    """
    conn.query(query)
    conn.commit()
    query = f"""
    INSERT INTO survey_data.survey_tags (survey_meta_id, tag_id, tag_value, survey_instance)
    SELECT st.survey_meta_id, mt.tag_id,
    FLOOR((st.tag_value - 1) / 3 * 100) AS tag_value,
     st.survey_instance
      FROM survey_data.survey_tags st,
           app.map_tag_id mt
     WHERE st.tag_id = 'multiplayer__multi'
       AND mt.id = {tag_id_int}
       AND st.survey_meta_id = {survey_id}
    """
    conn.query(query)
    conn.commit()

conn.commit()
conn.close()